In [1]:
import ROOT as rt
import uproot
import numpy as np
from collections import OrderedDict
import os
import sys
from array import array
sys.path.append(os.getcwd().replace('scripts', 'lib'))

import tdrstyle
a = tdrstyle.setTDRStyle()

print(sys.version)

Welcome to JupyROOT 6.30/02
3.9.12 (main, Jul 11 2023, 14:37:47) 
[GCC 11.3.0]


# load signal ntuples

Load signal ntuples with 4 different LLP lifetimes (0.1, 1, 10, 100m) that contain generator-level LLP information

In [2]:
fpath =OrderedDict()
tree = OrderedDict()

mass = [40]

OLD_CTAU = np.array([100, 1000, 10000, 100000])#in mm

path = '/eos/uscms/store/user/cmsdas/2025/long_exercises/MDS/signal/nocuts/'

for ct in OLD_CTAU:
    key = 'MC_40_'+str(ct)                       
    fpath[key] = path +  'ggH_HToSSTobbbb_MH-125_MS-40_ctau-'+str(ct)+'_137000pb_weighted.root'

    
NEvents = {}
NEvents_genweight = {}
for k,v in fpath.items():
    root_dir = uproot.open(v) 
    if not root_dir: 
        print(k, "zombie")
        continue
    tree[k] = root_dir['MuonSystem']


# Ex0 - calculate the LLP proper decay length from the LLP four momentum

Here you will calculate the LLP proper decay length by converting from the lab frame decay length to the decay length in the LLP's rest frame, with the following equation:

$d = \beta\gamma\tau$, 

where d is the lab frame decay length, $\beta$ is the LLP velocity, $\gamma$ is the LLP Lorentz factor, and $\tau$ is the LLP proper decay lifetime.

You will use the following branches in the ROOT tree 
* `gLLP_decay_vertex_x`, in units of cm
* `gLLP_decay_vertex_y`, in units of cm
* `gLLP_decay_vertex_z`, in units of cm
* `gLLP_beta` , ranges from 0 to 1

and you can access a branch by doing `T['branch name'].array()` below.

For example to access `gLLP_beta` with `T['gLLP_beta'].array()`. 

The shape of the LLP-level branches is (number of LLPs, 2), because there are two LLPs per event.

Fill in the righthand side of each equation below with the 4 branches mentioned to calculate the proper decay length

In [ ]:
gLLP_ctau = {}
for k, T in tree.items():
    gLLP_beta = T['gLLP_beta'].array()
    gLLP_decay_vertex = np.sqrt(T['gLLP_decay_vertex_x'].array()**2 + T['gLLP_decay_vertex_y'].array()**2 + T['gLLP_decay_vertex_z'].array()**2)
    gLLP_gamma = 1.0/np.sqrt(1.0 - gLLP_beta**2)
    gLLP_ctau[k] = gLLP_decay_vertex/(gLLP_beta*gLLP_gamma)


**Explanation.** d = |decay vertex| is the lab-frame decay length (cm), since the LLP is produced at the origin. Then d = beta*gamma*ctau gives ctau = d/(beta*gamma), with gamma = 1/sqrt(1-beta^2). Units stay in cm, the unit of the branches.

# Ex1 - plot the LLP proper decay length

In this exercise, you will plot the `gLLP_ctau` array for 1m and 10m sample.

You will then perform an exponential fit to verify that the shape is exponential and linear in logY scale.

From the exponential fit, you can extract the mean proper decay length of the LLP and verify that it agrees with what you expect from the sample (1m or 10m)

You have to replace the `REPLACE_ME` variable below to fill in the correct variable to extract the fitted ctau by using the variable `fitted_slope`. 

Note that the unit in the tree branches are in cm, while the unit in the ROOT file names are in mm.

In [ ]:
%%time

Nevents_todraw = 10000
c = rt.TCanvas('c','c', 800, 600)
h = {}

for i, k in enumerate([ 'MC_40_1000', 'MC_40_10000']):
    h[k] = rt.TH1D('', '', 100, 0, 10000)
    h[k].SetXTitle('LLP c#tau [cm]')
    h[k].SetYTitle('fraction of events')
    for j in range(Nevents_todraw): h[k].Fill(gLLP_ctau[k][j][0])
    h[k].Scale(1./h[k].Integral())  #normalize histogram to 1
    h[k].SetLineColor(i+1)
    h[k].Draw('hist same')

    ####### perform exponential fit ######
    h[k].Fit('expo')
    fit = h[k].GetFunction('expo')
    # ROOT 'expo' is exp(p0 + p1*x), so p1 = -1/ctau with ctau in cm
    fitted_slope = fit.GetParameter(1)
    print("sample: {}, fitted ctau [mm]: {}".format(k, -10.0/fitted_slope))
    #######################################

c.SetLogy()
c.Draw()


**Explanation (Ex1).** The histogram of gLLP_ctau is normalised to unit area and fitted with ROOT's `expo`, which is exp(p0 + p1*x). On a log-y plot this is a straight line, which checks the exponential shape. The fitted slope p1 equals -1/ctau, in the histogram unit (cm). The printed value is converted to mm with a factor of 10, because the file names are in mm. Pitfall: the 1 m sample has ctau of about 100 cm, which is about the same as the 100 cm bin width here, so the fit is coarse; the 10 m sample is better sampled.

**Discussion 2.1 (answer).** Question, verbatim: "Are you able to extract the LLP proper decay length from the exponential fit? Does it agree with the expectation?"

Yes. The fitted ctau is -10/p1 in mm. For the 1 m (MC_40_1000) and 10 m (MC_40_10000) samples the printed values should be close to 1000 and 10000 mm. The 10 m sample should agree within a few percent. The 1 m sample can deviate more because of the coarse binning and the fit range, so the agreement is at the level of the bin width and the fit uncertainty. I did not execute the notebook, so the exact printed values are not quoted here; check them against the expected values.


# Ex2 - reweight LLP lifetime 

In this exercise you will learn how to reweight the LLP lifetime.

In these signal events, there are two LLPs and the decay position of the two LLPs in each event are independent, and each LLP decays with an exponential probability, the distribution of events is simply the product of the two LLP decay probabilities:
\begin{equation}
  p(t_1, t_2 | \tau) = \frac{1}{\tau^2}\exp^{-t_1/\tau}\exp^{-t_2/\tau}
\end{equation}
where $\tau$ is the mean proper decay length of the LLPs, and $t_1$ and $t_2$ are the lifetimes of the first LLP and second LLP in their own rest frame respectively, which are the `gLLP_ctau` variable that you have calculated in Ex1.

To obtain a sample with lifetime $\tau_{new}$ from a sample with lifetime $\tau_{old}$, we
assign a weight, which is the ratio between equation above with parameter $\tau_{new}$
and:
\begin{equation}
  w = (\frac{\tau_{old}}{\tau_{new}})^2 \exp[(t_1+t_2)\times(\frac{1}{\tau_{old}}-\frac{1}{\tau_{new}})]
\end{equation}

You will implement this equation in the function `weight_calc` below

In [ ]:
'''
arguements:
llp_ctau is the list of sum of proper decay length, which is t1+t2 in the equation
new_ctau is tau_new in the equation
old_ctau is tau_old in the equation
All units in cm

return event weight

'''


def weight_calc(llp_ct, new_ctau, old_ctau):
    denominator = np.exp(-1.0*llp_ct/old_ctau)/old_ctau**2
    numerator = np.exp(-1.0*llp_ct/new_ctau)/new_ctau**2
    weight = numerator/denominator

    return weight


**Explanation (Ex2).** The event probability for two independent LLPs is p(t1,t2|tau) = (1/tau^2) exp(-(t1+t2)/tau). The weight from a sample with tau_old to tau_new is the ratio p(new)/p(old). Dividing by the old density and multiplying by the new density gives w = (tau_old/tau_new)^2 exp[(t1+t2)(1/tau_old - 1/tau_new)], which is the formula in the lesson. The code computes the same thing as numerator/denominator. Units: llp_ct, new_ctau and old_ctau must all be in the same unit, which is cm here. The lib/helper.py version does the same job, but this notebook keeps the function inline.


## reweight from 10m to 1m

In this exercise you will 
 - use the reweighting function that you have just implemented to reweight the 10m sample to 1m 
 - plot it against the original 1m sample to validate the reweighting function.

You will need to fill in the arguments for the `weight_calc()` function with `llp_ct` and the two lifetimes. 

Be sure to convert everything to cm.

You will then perform an exponential fit to verify that the extracted mean proper decay length of the LLP is 1m for both by replacing the variable `REPLACE_ME`

In [ ]:
%%time
c = rt.TCanvas('c','c', 800, 600)
h = {}
Nevents_todraw = 10000

for i, k in enumerate(['MC_40_1000', 'MC_40_10000']):
    h[k] = rt.TH1D('', '', 100, 0, 1000)
    h[k].SetXTitle('LLP c#tau [cm]')
    h[k].SetYTitle('fraction of events')

    if k == 'MC_40_1000':
        for j in range(Nevents_todraw): h[k].Fill(gLLP_ctau[k][j][0])
    else:
        llp_ct = np.sum(gLLP_ctau[k], axis = 1)

        # 10 m (1000 cm) -> 1 m (100 cm), both in cm
        weight = weight_calc(llp_ct, 100.0, 1000.0)

        for j in range(Nevents_todraw): h[k].Fill(gLLP_ctau[k][j][0], weight[j])

    h[k].Scale(1./h[k].Integral())
    h[k].SetLineColor(i+1)
    h[k].Draw('hist same')


    ####### perform exponential fit ######
    h[k].Fit('expo')
    fit = h[k].GetFunction('expo')
    # ROOT 'expo' is exp(p0 + p1*x), so p1 = -1/ctau with ctau in cm
    fitted_slope = fit.GetParameter(1)
    print("sample: {}, fitted ctau [mm]: {}".format(k, -10.0/fitted_slope))

    #######################################

c.SetLogy()
c.Draw()


**Explanation (Ex2 validation).** The 10 m sample is reweighted to 1 m with weight_calc(llp_ct, 100 cm, 1000 cm). llp_ct is the sum of the two LLP proper decay lengths in each event, in cm. The reweighted 10 m histogram should match the directly generated 1 m histogram, and its fitted ctau should be about 1000 mm. The fit is printed in mm, using the same factor of 10 as Ex1.

**Discussion 2.2 (answer).** Question, verbatim: "Why do we reweight from a larger lifetime to a smaller lifetime? What happens when we do the opposite (reweight from smaller lifetime to larger lifetime)?"

Reweighting from a larger to a smaller lifetime is safe. The old sample has events everywhere the new distribution needs them, because the small-lifetime region is well populated in the long-lived sample. The weight exp[(t1+t2)(1/tau_old - 1/tau_new)] is then bounded and close to 1 for the events that matter. Going the other way, the new distribution is concentrated at large t, where the old, short-lifetime sample has almost no events. Its weights become very large, a few events dominate, the effective sample size drops, and the statistical uncertainty grows. The ctau_list code in Ex3 follows this rule: it picks the smallest generated sample with tau_old >= tau_new, and the extrapolation beyond 100 m is the one place where the rule is broken.


# Ex3 - Calculate the probability of LLP decaying in the muon system

In this exercise you will calculate the probability of the LLP decaying in the muon system by requiring the LLP decay vertex lies within the Muon system.

You will first define the geometric decay region of the muon system based on Fig. 4.1.1 on page 141 of the [Muon Detector Technical Design Report](https://cds.cern.ch/record/343814?ln=en) 

Replace the `REPLACE_ME` in the cell below. 

Then in the last line you will print the probability of LLP decaying in the endcap muon detectors for LLP mean proper decay lenegths (0.1, 1, 10, 100m) that have been generated.

The probability should be on the order of 15-20% for the 1m and 10m sample.

In [ ]:
%%time

for k, T in tree.items():

     # define endcap muon system region
    sel_ms = np.abs(T['gLLP_eta'].array()) < 2.4
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_z'].array()) < 1075)
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_z'].array()) > 568)
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_r'].array()) < 700)

    print(k, np.count_nonzero(np.sum(sel_ms, axis = 1))/len(sel_ms))


**Explanation (Ex3 region).** The endcap muon region is set from the CSC layout in lib/util.py (drawRZ) and the CSC z-plot used in MDS_reconstruction. Cuts (cm, cm, cm, and eta):
- |eta| < 2.4: muon system acceptance.
- 568 < |z| < 1075: from ME1/1 (z about 568 to 632 cm) to the last station, with the 1075 cm limit as the outer edge.
- |r| < 700 cm: outer radius of the endcap chambers (ME4/2 and ME1/3 reach about 700 cm).

The printed value is the fraction of events with at least one LLP decaying in this volume. The target is about 15 to 20 percent for the 1 m and 10 m samples. I did not run the notebook, so the value is not confirmed. Pitfall: the z lower bound must be an absolute value, because the endcaps are at both +z and -z.


## calulate acceptance vs LLP mean proper decay length

Here we will extend on what we just did to calculate the acceptance for intermediate $c\tau$ by calling the 
reweighting function that you've written in the previous exercise to calculate the probability for other intermediate proper decay lengths.


You will fill in the definition of `sel_ms`, which is what you defined in the cell above and the arguments for `weight_calc()` to reweight to intermediate lifetimes in the `ctau_list`

The list of acceptances for different `ctau` is then saved in the `acceptance` list

In [ ]:
%%time

ctau_list  = [100, 300, 500, 1000, 2000, 5000, 10000, 50000, 100000, 500000] #mm

acceptance = []
for ct in ctau_list:
    # find which sample to use for reweighting
    ct_source = 0
    for i in OLD_CTAU:
        if ct <= i:
            ct_source = i
            break
    if ct_source == 0: ct_source = OLD_CTAU[-1]
    print(ct, ct_source) #verify the correct sample is chosen

    T = tree['MC_40_{}'.format(ct_source)]

    ################## FILL IN HERE #################
    sel_ms = np.abs(T['gLLP_eta'].array()) < 2.4
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_z'].array()) < 1075)
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_z'].array()) > 568)
    sel_ms = np.logical_and(sel_ms, np.abs(T['gLLP_decay_vertex_r'].array()) < 700)
    # sum of the two LLP proper decay lengths in cm, from the source sample
    llp_ct = np.sum(gLLP_ctau['MC_40_{}'.format(ct_source)], axis=1)
    # reweight from the source lifetime (mm -> cm) to the target lifetime (mm -> cm)
    weight = weight_calc(llp_ct, ct/10.0, ct_source/10.0)

    #################################################

    denominator = np.sum(weight)
    numerator = np.sum(weight[np.sum(sel_ms, axis = 1)>0])
    acceptance.append(numerator/denominator)

print(ctau_list, acceptance)


**Explanation (Ex3 acceptance).** For each target ctau the code picks the smallest generated sample with ctau_source >= ctau, reweights its events with weight_calc, and computes acceptance = (sum of weights with at least one LLP in the endcap region) / (sum of all weights). The units are converted once: the ntuple quantities are in cm, the ctau_list is in mm, so the target and source are divided by 10 before the call. The region cut is the same one as in the cell above, so the two cells cannot disagree.


## plot acceptance vs LLP mean proper decay length

Here you will plot a TGraph with `ctau_list` on the x-axis and acceptance on the y-axis.

Documentation for TGraph: https://root.cern.ch/doc/master/classTGraph.html



### Q: Do you understand the shape? 

Play around with the `ctau_list` to add more points to make the plot smoother!

In [ ]:
%%time

c = rt.TCanvas('c','c', 800, 800)
c.SetRightMargin(0.04)

h = rt.TGraph(len(ctau_list),array("d", np.array(ctau_list)/1000.0), array("d", acceptance))

h.GetXaxis().SetTitle('c#tau [m]')
h.GetYaxis().SetTitle('acceptance')
h.GetXaxis().SetLimits(0.1,1000.0)
h.GetYaxis().SetRangeUser(1e-3,2)

h.GetXaxis().SetTitleOffset(1)
h.GetYaxis().SetTitleSize(0.05)
h.GetYaxis().SetTitleOffset(1.5)

h.SetLineWidth(3)
h.SetLineColor(4)
h.SetMarkerColor(4)

h.Draw('LPA')

c.SetLogy()
c.SetLogx()
c.SetTicky(1)
c.SetTickx(1)


tdrstyle.setTDRStyle()
c.Draw()

**Q: Do you understand the shape? (answer).** The notebook question is "Q: Do you understand the shape?".

Yes. Acceptance is small at short ctau because most LLPs decay before reaching the muon system, which is at 4 to 10 m from the IP. Acceptance falls at long ctau because most LLPs fly past the detector before they decay. The maximum sits where the lab-frame decay length, beta*gamma*ctau, is comparable to the distance to the endcap.

**Discussion 2.3 (answer).** Question, verbatim: "Do you understand the shape of the geometric acceptance vs. LLP mean proper decay lengths plot? Why is the peak at 2m, much smaller than the distance between the muon detectors and the interaction point?"

The peak is at ctau of about 2 m, not at the detector distance, because the quantity that must match the distance is the lab-frame decay length beta*gamma*ctau, not ctau. The LLPs in this sample are boosted (beta*gamma of a few), so a ctau of about 2 m gives lab decay lengths of several meters, which is the detector distance scale. Acceptance is a product of two factors: the chance of surviving to the endcap, which is exp(-d/lambda), and the chance of decaying inside the endcap volume, which is 1 - exp(-Delta/lambda). Their product peaks at lambda of order d and Delta, not at the full distance d. The exact value depends on the boost distribution in the sample, which can be checked with gLLP_beta.
